# Algoritmos de Clasificación

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from funciones_mecanizados import normalizar

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [2]:
fresado_h1 = pd.read_csv('data/fresado_H1_norm.csv')
fresado_h2 = pd.read_csv('data/fresado_H2_norm.csv')
fresado_h3 = pd.read_csv('data/fresado_H3_norm.csv')

In [3]:
fresado = pd.concat([fresado_h1, fresado_h2, fresado_h3])
fresado = shuffle(fresado)

In [4]:
# Pasamos la variable categórica 'Herramienta' a una variable numérica.
fresado.Herramienta.replace({'Nueva':3, 'Usada':2, 'Muy Usada':1}, inplace=True)

## Train/Test split

In [5]:
from sklearn.model_selection import StratifiedShuffleSplit

train_test_split = StratifiedShuffleSplit(n_splits=1, test_size=.20, random_state=11)
for train_index, test_index in train_test_split.split(fresado, fresado['Herramienta']):
    X_train = fresado.iloc[train_index]
    X_test = fresado.iloc[test_index]

In [6]:
print('Datos totales:',fresado.shape[0])
print('Train:        ',X_train.shape[0])
print('Test:         ',X_test.shape[0])

Datos totales: 739
Train:         591
Test:          148


In [7]:
X = X_train.copy()
y = X.pop('Herramienta')
X.drop(['CH','Tiempo', 'Mecanizado'], axis=1, inplace=True)

## Random Forest Classifier

In [8]:
grilla_parametros =  {'n_estimators': [150,400],
              'max_features': [5, 7, 9, 12, 15],
              'max_depth'   : [None, 3, 10, 20],
              'criterion'   : ['gini', 'entropy']
             }


kf= StratifiedShuffleSplit(n_splits=5, test_size=.20, random_state=11)

clasificador = GridSearchCV(RandomForestClassifier(random_state = 14),
                            param_grid=grilla_parametros, 
                            cv=kf,
                            verbose=2, 
                            n_jobs=2
                           )
clasificador.fit(X, y)

print(clasificador.best_params_, ":", clasificador.best_score_)

Fitting 5 folds for each of 80 candidates, totalling 400 fits


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:   22.2s
[Parallel(n_jobs=2)]: Done 158 tasks      | elapsed:  1.4min
[Parallel(n_jobs=2)]: Done 361 tasks      | elapsed:  3.4min
[Parallel(n_jobs=2)]: Done 400 out of 400 | elapsed:  4.0min finished


{'criterion': 'entropy', 'max_depth': None, 'max_features': 5, 'n_estimators': 400} : 0.9310924369747899


In [9]:
cols = ['RMS','AMP','ENER','P_FRQ','PCNTS','C_FRQ', 'I_FRQ','A_FRQ',
        'R_FRQ','DURATION','ASL','COUN','ABS_ENERGY', 'SIG_STRNGTH','RISE']

y_train = X_train.Herramienta
y_test = X_test.Herramienta

X_train = X_train[cols]
X_test = X_test[cols]

In [10]:
forest_model = RandomForestClassifier(n_estimators=clasificador.best_params_['n_estimators'], 
                                      max_features=clasificador.best_params_['max_features'], 
                                      criterion=clasificador.best_params_['criterion'], 
                                      max_depth=None)
forest_model.fit(X_train, y_train)
prediccion = forest_model.predict(X_test)
print(accuracy_score(y_test, prediccion))

0.9662162162162162
